In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Voltage-Aware Network Clustering with NPAP

This example compares PyPSA's coordinate-based K-means clustering with voltage-aware clustering through the optional [`npap`](https://npap.readthedocs.io) package. To run it locally, install PyPSA with the `npap` optional dependency. The example uses the SciGRID-DE network and asks both methods for 50 clusters.

The relevant difference is that the NPAP strategy partitions each voltage level separately. This prevents a reduced bus from representing both 220 kV and 380 kV buses, which the coordinate-only K-means method cannot enforce directly.

In [ ]:
import logging

import pandas as pd

import pypsa

warnings.filterwarnings("ignore", message="pandas infers the `str` dtype")
warnings.filterwarnings("ignore", message="Parallel edges detected.*")
warnings.filterwarnings("ignore", message="No numeric values found.*")

logging.getLogger("pypsa").setLevel(logging.WARNING)
logging.getLogger("npap").setLevel(logging.ERROR)
logging.getLogger("pypsa.clustering.spatial").setLevel(logging.ERROR)
logging.getLogger("pypsa.network.transform").setLevel(logging.ERROR)

N_CLUSTERS = 50
VOLTAGE_LEVELS = [220, 380]
NPAP_STRATEGY = "va_geographical_kmedoids_haversine"

## Load the Network

SciGRID-DE contains both 220 kV and 380 kV buses, connected by transformers. That makes it a compact example for checking whether a clustering method respects voltage-level boundaries.

In [ ]:
n = pypsa.examples.scigrid_de()
n.calculate_dependent_values()
n.determine_network_topology()

n.buses.v_nom.value_counts().sort_index().rename("buses")

## Coordinate-Based K-means

PyPSA's K-means helper clusters buses based on their coordinates. We use uniform bus weights here so the comparison focuses on the partitioning constraint rather than on weighting choices.

In [ ]:
bus_weightings = pd.Series(1, index=n.buses.index)

busmap_kmeans = n.cluster.spatial.busmap_by_kmeans(
    bus_weightings=bus_weightings,
    n_clusters=N_CLUSTERS,
    random_state=0,
)

## Voltage-Aware NPAP Clustering

The NPAP strategy used below is also geographical, but it first separates the network into voltage-aware groups. In this case, the target levels are the two transmission levels present in SciGRID-DE.

In [ ]:
busmap_npap = n.cluster.spatial.busmap_by_npap(
    n_clusters=N_CLUSTERS,
    strategy=NPAP_STRATEGY,
    voltage_levels=VOLTAGE_LEVELS,
    random_state=0,
)

## Compare Voltage Mixing

A mixed-voltage cluster is a cluster whose original buses contain more than one nominal voltage level.

In [ ]:
def mixed_voltage_cluster_count(busmap):
    voltage_levels_per_cluster = (
        n.buses[["v_nom"]].assign(cluster=busmap).groupby("cluster").v_nom.nunique()
    )
    return int(voltage_levels_per_cluster.gt(1).sum())


comparison = pd.DataFrame(
    {
        "clusters": [busmap_kmeans.nunique(), busmap_npap.nunique()],
        "mixed_voltage_clusters": [
            mixed_voltage_cluster_count(busmap_kmeans),
            mixed_voltage_cluster_count(busmap_npap),
        ],
    },
    index=["PyPSA K-means", "NPAP voltage-aware k-medoids"],
)
comparison

With the SciGRID-DE example and `random_state=0`, K-means creates 49 mixed-voltage clusters, while the NPAP voltage-aware strategy creates 0. This is the main capability difference: voltage-aware NPAP clustering can enforce an electrical grouping rule that the coordinate-only K-means interface cannot express directly.

## Return a PyPSA Network

The same strategy can be used directly through `cluster_by_npap()`. The returned object is a regular clustered PyPSA network.

In [ ]:
n_clustered = n.cluster.spatial.cluster_by_npap(
    n_clusters=N_CLUSTERS,
    strategy=NPAP_STRATEGY,
    voltage_levels=VOLTAGE_LEVELS,
    random_state=0,
    aggregate_one_ports=["Generator", "Load", "StorageUnit"],
    with_time=False,
)

pd.Series(
    {
        "type": type(n_clustered).__name__,
        "buses": len(n_clustered.buses),
    }
)